# CS224N · Lecture 9 — Pretraining

**Slides:** [cs224n-spr2024-lecture09-pretraining-updated.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture09-pretraining-updated.pdf) (Christopher Manning; adapted from slides by Anna Goldie, John Hewitt, Tatsunori Hashimoto)  
**Prerequisites:** L8 (the Transformer; this notebook reuses the NumPy Transformer from L8, refactored so the same backbone can carry different "heads").

### The pretraining revolution
Pretraining has had a major, tangible impact on how well NLP systems work. The key ideas:
* Make sure your model can process **large-scale, diverse datasets**.
* **Don't use labeled data** (otherwise you can't scale!).
* **Compute-aware scaling.**

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Subword modeling | **Byte-pair encoding** from scratch, on the slide's example and a small corpus |
| 2 | Motivating pretraining: from word embeddings to whole models | — |
| 3 | **Pretrain → finetune**, measured | Pretrain a decoder LM, then compare zero-shot scoring, training from scratch, a linear probe, and full finetuning on a low-data task |
| 4 | Pretraining **encoders**: masked LM (BERT) | BERT's 80/10/10 masking; an encoder that uses the *right* context |
| 5 | Pretraining **encoder–decoders**: span corruption (T5) | T5's span-corruption preprocessing |
| 6 | Pretraining **decoders**: GPT, GPT-2, GPT-3, in-context learning | — |
| 7 | **Scaling laws** and compute-optimal training (Chinchilla) | Compute-optimal model/data sizes from the Chinchilla fit |
| 8 | What does pretraining teach? (and what we don't want it to learn) | — |
| 9 | Hugging Face versions (optional) | BERT fill-mask, GPT-2 scoring, T5 sentinels |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections, re

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

# ---- the NumPy Transformer from L8, refactored into a backbone + separate heads ----
def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def ln_forward(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    xh = (x - mu) / np.sqrt(var + eps)
    return xh * g + b, (xh, var, eps)

def ln_backward(dy, g, cache):
    xh, var, eps = cache; D = xh.shape[-1]
    dg = (dy * xh).reshape(-1, D).sum(0); db = dy.reshape(-1, D).sum(0)
    dxh = dy * g
    dx = (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True)) / np.sqrt(var + eps)
    return dx, dg, db

def init_transformer(V, T_max, d=48, n_layers=2, n_heads=4, d_ff=96, seed=0):
    r = np.random.default_rng(seed)
    p = {"tok": r.normal(0, 0.02, (V, d)), "pos": r.normal(0, 0.02, (T_max, d))}
    for l in range(n_layers):
        for k in ["Wq", "Wk", "Wv", "Wo"]:
            p[f"{l}.{k}"] = r.normal(0, 1 / np.sqrt(d), (d, d))
        p[f"{l}.W1"], p[f"{l}.b1"] = r.normal(0, 1 / np.sqrt(d), (d, d_ff)), np.zeros(d_ff)
        p[f"{l}.W2"], p[f"{l}.b2"] = r.normal(0, 1 / np.sqrt(d_ff), (d_ff, d)), np.zeros(d)
        for k in ["ln1", "ln2"]:
            p[f"{l}.{k}g"], p[f"{l}.{k}b"] = np.ones(d), np.zeros(d)
    p["lnfg"], p["lnfb"] = np.ones(d), np.zeros(d)
    return p

def mha_forward(a, Wq, Wk, Wv, Wo, nh, causal, key_mask):
    B, T, d = a.shape; dk = d // nh
    split = lambda z: z.reshape(B, T, nh, dk).transpose(0, 2, 1, 3)
    q, k, v = split(a @ Wq), split(a @ Wk), split(a @ Wv)
    S = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dk)
    if causal:
        S = np.where(np.triu(np.ones((T, T), bool), 1), -1e9, S)
    if key_mask is not None:                                   # (B, T): True = real token
        S = np.where(key_mask[:, None, None, :], S, -1e9)
    P = softmax(S, -1)
    Zc = (P @ v).transpose(0, 2, 1, 3).reshape(B, T, d)
    return Zc @ Wo, (a, q, k, v, P, Zc, dk)

def mha_backward(dout, Wq, Wk, Wv, Wo, nh, cache):
    a, q, k, v, P, Zc, dk = cache; B, T, d = a.shape
    gWo = Zc.reshape(-1, d).T @ dout.reshape(-1, d)
    dZ = (dout @ Wo.T).reshape(B, T, nh, dk).transpose(0, 2, 1, 3)
    dP = dZ @ v.transpose(0, 1, 3, 2); dv = P.transpose(0, 1, 3, 2) @ dZ
    dS = P * (dP - (dP * P).sum(-1, keepdims=True)) / np.sqrt(dk)
    dq, dk_ = dS @ k, dS.transpose(0, 1, 3, 2) @ q
    merge = lambda z: z.transpose(0, 2, 1, 3).reshape(B, T, d)
    dq, dk_, dv = merge(dq), merge(dk_), merge(dv)
    a2 = a.reshape(-1, d)
    return dq @ Wq.T + dk_ @ Wk.T + dv @ Wv.T, (a2.T @ dq.reshape(-1, d), a2.T @ dk_.reshape(-1, d), a2.T @ dv.reshape(-1, d), gWo)

def backbone_forward(p, X, n_layers, n_heads, causal=True, key_mask=None):
    # Pre-LN Transformer. Returns final hidden states (B, T, d) and a cache for the backward pass.
    B, T = X.shape
    x = p["tok"][X] + p["pos"][:T]
    caches, attn = [], []
    for l in range(n_layers):
        a_in, c1 = ln_forward(x, p[f"{l}.ln1g"], p[f"{l}.ln1b"])
        att, ca = mha_forward(a_in, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads, causal, key_mask)
        attn.append(ca[4]); x1 = x + att
        f_in, c2 = ln_forward(x1, p[f"{l}.ln2g"], p[f"{l}.ln2b"])
        hpre = f_in @ p[f"{l}.W1"] + p[f"{l}.b1"]; hid = np.maximum(hpre, 0)
        x = x1 + hid @ p[f"{l}.W2"] + p[f"{l}.b2"]
        caches.append((c1, ca, c2, f_in, hpre, hid))
    xf, cf = ln_forward(x, p["lnfg"], p["lnfb"])
    return xf, (X, caches, cf, attn)

def backbone_backward(p, dxf, cache, n_layers, n_heads):
    X, caches, cf, _ = cache; g = {}; d = dxf.shape[-1]
    dx, g["lnfg"], g["lnfb"] = ln_backward(dxf, p["lnfg"], cf)
    for l in reversed(range(n_layers)):
        c1, ca, c2, f_in, hpre, hid = caches[l]
        g[f"{l}.W2"] = hid.reshape(-1, hid.shape[-1]).T @ dx.reshape(-1, d); g[f"{l}.b2"] = dx.reshape(-1, d).sum(0)
        dhpre = (dx @ p[f"{l}.W2"].T) * (hpre > 0)
        g[f"{l}.W1"] = f_in.reshape(-1, d).T @ dhpre.reshape(-1, dhpre.shape[-1]); g[f"{l}.b1"] = dhpre.reshape(-1, dhpre.shape[-1]).sum(0)
        dx1_ln, g[f"{l}.ln2g"], g[f"{l}.ln2b"] = ln_backward(dhpre @ p[f"{l}.W1"].T, p[f"{l}.ln2g"], c2)
        dx1 = dx + dx1_ln
        da, (g[f"{l}.Wq"], g[f"{l}.Wk"], g[f"{l}.Wv"], g[f"{l}.Wo"]) = mha_backward(
            dx1, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads, ca)
        dxa, g[f"{l}.ln1g"], g[f"{l}.ln1b"] = ln_backward(da, p[f"{l}.ln1g"], c1)
        dx = dx1 + dxa
    g["tok"] = np.zeros_like(p["tok"]); np.add.at(g["tok"], X, dx)
    g["pos"] = np.zeros_like(p["pos"]); g["pos"][:X.shape[1]] = dx.sum(0)
    return g

def token_ce(logits, Y, M):
    # Cross-entropy over positions with mask M. Returns loss, dlogits.
    P = softmax(logits, -1); B, T = Y.shape
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(T), indexing="ij"); n = max(M.sum(), 1)
    loss = -(np.log(P[Bi, Ti, Y] + 1e-12) * M).sum() / n
    G = P.copy(); G[Bi, Ti, Y] -= 1; G *= M[:, :, None] / n
    return loss, G, P

class Adam:
    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, keys=None):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.keys = list(params) if keys is None else list(keys)          # which parameters to train
        self.m = {k: np.zeros_like(params[k]) for k in self.keys}
        self.v = {k: np.zeros_like(params[k]) for k in self.keys}
    def step(self, params, grads):
        self.t += 1
        for k in self.keys:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def clip_grads(g, max_norm=1.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm

---
## 1. A brief note on subword modeling

So far we assumed a **fixed vocabulary** of tens of thousands of words, built from the training set. All novel words seen at test time are mapped to a single **UNK**:

| word | vocab mapping |
|---|---|
| hat | → hat (common word) |
| learn | → learn (common word) |
| taaaaasty | → **UNK** (variation) |
| laern | → **UNK** (misspelling) |
| Transformerify | → **UNK** (novel item) |

Finite-vocabulary assumptions make even less sense in many languages other than English. Many languages have **complex morphology** (word structure), so there are many more word types, each occurring fewer times. Example: **Swahili verbs can have hundreds of conjugations**, each encoding tense, mood, definiteness, negation, information about the object, and more (the slide shows a small fraction of the forms of *ambia*, "to tell").

### 1.1 The byte-pair encoding (BPE) algorithm
**Subword modeling** covers a wide range of methods for reasoning about structure below the word level (parts of words, characters, bytes). The dominant modern paradigm is to **learn a vocabulary of parts of words** (subword tokens); at training and testing time, each word is split into a sequence of known subwords.

**Byte-pair encoding** (Sennrich et al. 2016; originally a compression algorithm) is a simple, effective strategy:
1. Start with a vocabulary containing only **characters** and an **"end-of-word"** symbol.
2. Using a corpus of text, find the **most common adjacent pair** "a, b"; add "ab" as a subword.
3. **Replace** instances of the pair with the new subword; repeat until the desired vocabulary size.

Originally used for MT; now similar methods (WordPiece, SentencePiece) are used in pretrained models like BERT and GPT.

The slides' example: a dictionary with word counts **low ×5, lower ×2, newest ×6, widest ×3**. The first merges are `e s → es` (9 occurrences), then `es t → est`.

In [ ]:
EOW = "</w>"

def pair_counts(vocab):
    # vocab: {tuple of symbols: count}. Count adjacent symbol pairs, weighted by word frequency.
    counts = collections.Counter()
    for word, freq in vocab.items():
        for a, b in zip(word, word[1:]):
            counts[(a, b)] += freq
    return counts

def merge_pair(vocab, pair):
    merged = {}
    for word, freq in vocab.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                out.append(word[i] + word[i + 1]); i += 2
            else:
                out.append(word[i]); i += 1
        merged[tuple(out)] = freq
    return merged

def learn_bpe(word_freqs, num_merges, verbose=False):
    vocab = {tuple(w) + (EOW,): f for w, f in word_freqs.items()}
    merges = []
    for step in range(num_merges):
        counts = pair_counts(vocab)
        if not counts:
            break
        top = max(counts.values())
        best = sorted(p for p, c in counts.items() if c == top)[0]       # most frequent pair; ties broken alphabetically
        if verbose:
            print(f"merge {step + 1}: {best[0]} + {best[1]} -> {best[0] + best[1]}   (count {counts[best]})")
        vocab = merge_pair(vocab, best)
        merges.append(best)
    return merges, vocab

slide_dict = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
merges, final_vocab = learn_bpe(slide_dict, 8, verbose=True)
print("\nwords as subword sequences after 8 merges:")
for w, f in final_vocab.items():
    print(f"  {f} x {' '.join(w)}")

**Applying BPE to a new word:** split it into characters (+ end-of-word), then apply the learned merges **in the order they were learned**.

Common words end up being a part of the subword vocabulary, while rarer words are split into (sometimes intuitive, sometimes not) components. In the worst case, words are split into as many subwords as they have characters. Let's learn a few hundred merges from a small corpus and look at how it handles the slide's examples:

In [ ]:
def apply_bpe(word, merges):
    symbols = list(word) + [EOW]
    rank = {m: i for i, m in enumerate(merges)}
    while True:
        pairs = [(rank.get((a, b), math.inf), i) for i, (a, b) in enumerate(zip(symbols, symbols[1:]))]
        best_rank, i = min(pairs, default=(math.inf, None))
        if best_rank == math.inf:
            return symbols
        symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

corpus = '''
the students learned to transform the data before they learned the model . a tasty meal is a meal that
tastes good . the learner learns while the teacher teaches . transformers transform sequences into
representations . the hat was on the table near the tasty pizza . we learn from data and we keep
learning . the newest model is the lowest cost model and the widest model . lower the learning rate
when the loss is unstable . the transformer learned to translate , summarize and classify . people
who learn languages learn new words every day . the tastiest pizza in town has the thinnest crust .
she transformed the room ; he is transforming the garden . learners learned , learning continues .
'''.split()
word_freqs = collections.Counter(corpus)
merges_c, vocab_c = learn_bpe(word_freqs, 150)
for w in ["hat", "learn", "learning", "taaaaasty", "laern", "transformerify", "lowest"]:
    print(f"{w:16s} -> {' '.join(apply_bpe(w, merges_c))}")

Frequent words become single tokens (`learn</w>`), related words share pieces (`learn` + `ing`), and novel or misspelled words fall back to smaller pieces instead of UNK. (BERT's WordPiece marks *continuation* pieces with `##` instead of marking the end of a word: `transform ##er ##ify`.)

🔎 **What modern tokenizers actually do:**
* **Byte-level BPE** (GPT-2 onward): start from the 256 **bytes** instead of characters, so *any* string (any language, emoji, code) can be encoded and there is never an UNK.
* **WordPiece** (BERT): like BPE, but chooses merges by how much they increase the likelihood of the training data, not by raw frequency.
* **SentencePiece** (T5, LLaMA): treats the input as a raw character stream with whitespace as a symbol (`▁`), so no pre-tokenization into words is needed; it supports BPE and the **unigram LM** tokenizer.
* Tokenization has real consequences: the same text can cost several times more tokens in some languages than in English (higher cost and shorter effective context), and tokenization of numbers affects arithmetic ability.

### 1.2 Words in writing systems
Writing systems vary in how they represent words, or don't:
* **No word segmentation:** 安理会认可利比亚问题柏林峰会成果 (Chinese)
* **Words (mainly) segmented:** *This is a sentence with words.*
* **Clitics/pronouns/agreement:** separated (*Je vous ai apporté des bonbons*) or joined (Arabic *faqulnāhā* = so + we said + it)
* **Compounds:** separated (*life insurance company employee*) or joined (*Lebensversicherungsgesellschaftsangestellter*)

Below the word level, scripts can be phonemic (Wambaya), "fossilized phonemic" (English *thorough failure*), syllabic (Inuktitut ᑐᖑᔪᐊᖓᔪᖅ), ideographic (Chinese), or a combination (Japanese インド洋の島). Subword and byte-level tokenizers are one reason a single model can now handle all of these.

---
## 2. Motivating model pretraining from word embeddings

Recall the adage from the beginning of the course: *"You shall know a word by the company it keeps"* (J. R. Firth 1957). This summary of distributional semantics motivated word2vec. But Firth also wrote:

> *"… the complete meaning of a word is always contextual, and no study of meaning apart from a complete context can be taken seriously."* (J. R. Firth 1935)

Consider ***I record the record***: the two instances of *record* mean different things.

**Where we were (circa 2015):** start with **pretrained word embeddings** (no context!), and learn how to incorporate context in an LSTM or Transformer while training on the task. Issues:
* The training data for our downstream task (like question answering) must be sufficient to teach **all contextual aspects of language**.
* **Most of the parameters in our network are randomly initialized!**

**Where we're going: pretraining whole models.** In modern NLP:
* All (or almost all) parameters in NLP networks are **initialized via pretraining**.
* Pretraining methods **hide parts of the input from the model, and train the model to reconstruct those parts**.
* This has been exceptionally effective at building strong **representations of language**, **parameter initializations** for strong NLP models, and **probability distributions over language** that we can sample from.

### 2.1 What can we learn from reconstructing the input?
| Fill in the blank | What it teaches |
|---|---|
| *Stanford University is located in \_\_\_\_, California.* | trivia / world knowledge |
| *I put \_\_\_ fork down on the table.* | syntax |
| *The woman walked across the street, checking for traffic over \_\_\_ shoulder.* | coreference |
| *I went to the ocean to see the fish, turtles, seals, and \_\_\_.* | lexical semantics / topic |
| *Overall, the value I got from the two hours watching it was the sum total of the popcorn and the drink. The movie was \_\_\_.* | sentiment |
| *Iroh went into the kitchen to make some tea. Standing next to Iroh, Zuko pondered his destiny. Zuko left the \_\_\_.* | some reasoning (harder) |
| *I was thinking about the sequence that goes 1, 1, 2, 3, 5, 8, 13, 21, \_\_\_* | some basic arithmetic (they don't learn the Fibonacci sequence) |

### 2.2 Pretraining through language modeling (Dai & Le 2015)
Recall the language modeling task: model $p_\theta(w_t\mid w_{1:t-1})$. There's lots of data for this (in English)! **Pretraining through language modeling:** train a neural network to perform language modeling on a large amount of text, and **save the network parameters**.

### 2.3 The pretraining / finetuning paradigm
Pretraining can improve NLP applications by serving as **parameter initialization**.
* **Step 1: Pretrain** (on language modeling). Lots of text; learn general things!
* **Step 2: Finetune** (on your task). Not many labels; adapt to the task!

**Why should this help, from a "training neural nets" perspective?**
* Pretraining provides parameters $\hat\theta$ by approximating $\min_\theta\mathcal{L}_{\text{pretrain}}(\theta)$.
* Then finetuning approximates $\min_\theta\mathcal{L}_{\text{finetune}}(\theta)$, **starting at $\hat\theta$**.
* The pretraining may matter because SGD **sticks (relatively) close to $\hat\theta$** during finetuning. So maybe the finetuning local minima near $\hat\theta$ tend to generalize well, and/or maybe the gradients of the finetuning loss near $\hat\theta$ propagate nicely!

### 2.4 Where does this data come from?
| Model | Training data |
|---|---|
| BERT | BookCorpus, English Wikipedia |
| GPT-1 | BookCorpus |
| GPT-3 | CommonCrawl, WebText, English Wikipedia, and 2 book databases ("Books 1" and "Books 2") |
| GPT-3.5+ | Undisclosed |

**BookCorpus** consists of ebooks scraped from the internet, which is highly controversial. Pretraining data raises **fair use and other copyright concerns**, as well as privacy concerns (§8).

---
## 3. Pretrain → finetune, measured

Let's see the paradigm work at small scale. We use the artificial language from L5 and L8, with long-distance subject–verb agreement across attractors.
* **Pretraining data (unlabeled):** 6,000 sentences, used only for language modeling.
* **Downstream task (labeled, low-resource):** **grammaticality judgment**, like the CoLA task in GLUE. Half of the sentences are correct; in the other half, the main verb's number is flipped (*the dogs that the cat saw near the farmer **sleeps***). We only get **200 labeled examples** for training.

In [ ]:
NOUNS = [("dog", "dogs"), ("cat", "cats"), ("pilot", "pilots"), ("farmer", "farmers"), ("author", "authors"), ("key", "keys")]
VERBS = [("sleeps", "sleep"), ("laughs", "laugh"), ("waits", "wait"), ("smiles", "smile")]
TRANS = ["saw", "liked", "chased", "met", "helped"]
ADJ = ["old", "young", "tall", "small", "happy"]
PREP = ["near", "behind", "with"]

def gen_sentence(max_mods=3, rng=random):
    plural = rng.random() < 0.5
    w = ["the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else [])
    w.append(rng.choice(NOUNS)[plural]); subj = len(w) - 1
    for _ in range(rng.choice(range(max_mods + 1))):
        ap = rng.random() < 0.5
        if rng.random() < 0.5:
            w += ["that", "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else []) + [rng.choice(NOUNS)[ap], rng.choice(TRANS)]
        else:
            w += [rng.choice(PREP), "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else []) + [rng.choice(NOUNS)[ap]]
    vpair = rng.choice(VERBS); w.append(vpair[plural]); verb = len(w) - 1
    if rng.random() < 0.3:
        w.append("quietly")
    w.append(".")
    return w, subj, verb, plural, vpair

def flip_verb(s, verb, plural, vpair):
    s2 = list(s); s2[verb] = vpair[1 - plural]
    return s2

random.seed(0)
pretrain_sents = [gen_sentence() for _ in range(6000)]
def labeled_example(rng=random):
    s, sp, vp, pl, vpair = gen_sentence(rng=rng)
    if rng.random() < 0.5:
        return s, 1, (s, vp, pl, vpair)
    return flip_verb(s, vp, pl, vpair), 0, (s, vp, pl, vpair)
task_train = [labeled_example() for _ in range(200)]
task_test = [labeled_example() for _ in range(1000)]

vocab = ["<pad>", "<s>", "</s>", "[MASK]"] + sorted({w for s, *_ in pretrain_sents for w in s})
w2i = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
T_MAX = 32
for s, y, _ in task_train[:4]:
    print(("✓ " if y else "✗ ") + " ".join(s))

### 3.1 Step 1: pretrain a decoder (a language model) on the unlabeled text
The backbone is L8's pre-LN Transformer (2 layers, 4 heads, $d = 48$). The **LM head** is a matrix $W_{\text{out}}$ mapping hidden states to vocabulary logits. We gradient-check the refactored backbone together with both heads we'll use, the LM head and a classification head on the last token.

In [ ]:
N_LAYERS, N_HEADS = 2, 4

def lm_batch(sents):
    L = max(len(s) for s in sents) + 1
    X = np.zeros((len(sents), L), int); Y = np.zeros((len(sents), L), int); M = np.zeros((len(sents), L))
    for b, s in enumerate(sents):
        ids = [w2i["<s>"]] + [w2i[w] for w in s] + [w2i["</s>"]]
        X[b, :len(ids) - 1] = ids[:-1]; Y[b, :len(ids) - 1] = ids[1:]; M[b, :len(ids) - 1] = 1
    return X, Y, M

def lm_loss_and_grads(p, X, Y, M):
    xf, cache = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=True, key_mask=M > 0)
    loss, G, _ = token_ce(xf @ p["Wout"], Y, M)
    g = backbone_backward(p, G @ p["Wout"].T, cache, N_LAYERS, N_HEADS)
    g["Wout"] = xf.reshape(-1, xf.shape[-1]).T @ G.reshape(-1, G.shape[-1])
    return loss, g

def cls_batch(examples):
    # Classification input: <s> w1 .. wn ; the classifier reads the hidden state of the LAST real token.
    L = max(len(s) for s, *_ in examples) + 1
    X = np.zeros((len(examples), L), int); M = np.zeros((len(examples), L)); last = np.zeros(len(examples), int)
    y = np.array([lab for _, lab, *_ in examples])
    for b, (s, *_) in enumerate(examples):
        ids = [w2i["<s>"]] + [w2i[w] for w in s]
        X[b, :len(ids)] = ids; M[b, :len(ids)] = 1; last[b] = len(ids) - 1
    return X, M, last, y

def cls_loss_and_grads(p, X, M, last, y, need_grad=True):
    xf, cache = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=True, key_mask=M > 0)
    h = xf[np.arange(len(y)), last]                       # (B, d): last word's hidden state
    P = softmax(h @ p["A"] + p["c"], -1)
    loss = -np.log(P[np.arange(len(y)), y] + 1e-12).mean()
    if not need_grad:
        return loss, P
    G = P.copy(); G[np.arange(len(y)), y] -= 1; G /= len(y)
    dxf = np.zeros_like(xf); dxf[np.arange(len(y)), last] = G @ p["A"].T
    g = backbone_backward(p, dxf, cache, N_LAYERS, N_HEADS)
    g["A"], g["c"] = h.T @ G, G.sum(0)
    return loss, g

# gradient check (tiny model, both heads)
r = np.random.default_rng(1)
saved = (N_LAYERS, N_HEADS)
N_LAYERS, N_HEADS = 2, 2
p_chk = init_transformer(V, T_MAX, d=8, n_layers=2, n_heads=2, d_ff=12, seed=1)
for k in p_chk:
    p_chk[k] = p_chk[k] + r.normal(0, 0.2, p_chk[k].shape)
p_chk["Wout"] = r.normal(0, 0.3, (8, V)); p_chk["A"] = r.normal(0, 0.3, (8, 2)); p_chk["c"] = np.zeros(2)
Xl, Yl, Ml = lm_batch([s for s, *_ in pretrain_sents[:2]])
Xc, Mc, lc, yc = cls_batch(task_train[:3])
worst = 0.0
for name, f in [("LM", lambda: lm_loss_and_grads(p_chk, Xl, Yl, Ml)), ("CLS", lambda: cls_loss_and_grads(p_chk, Xc, Mc, lc, yc))]:
    _, g = f()
    for k in g:
        for _ in range(6):
            idx = tuple(r.integers(0, s_) for s_ in p_chk[k].shape)
            if k == "tok":
                idx = (Xl[0, 1] if name == "LM" else Xc[0, 1], r.integers(0, 8))
            old = p_chk[k][idx]
            p_chk[k][idx] = old + 1e-6; lp = f()[0]
            p_chk[k][idx] = old - 1e-6; lm_ = f()[0]
            p_chk[k][idx] = old
            worst = max(worst, abs((lp - lm_) / 2e-6 - g[k][idx]))
print(f"gradient check, LM head and classification head: max |analytic - numerical| = {worst:.1e}")
N_LAYERS, N_HEADS = saved

In [ ]:
p_pre = init_transformer(V, T_MAX, d=48, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=96, seed=0)
p_pre["Wout"] = np.random.default_rng(0).normal(0, 1 / np.sqrt(48), (48, V))
opt = Adam(p_pre, lr=1e-3); r = np.random.default_rng(0); t0 = time.time()
for step in range(1, 1201):
    X, Y, M = lm_batch([pretrain_sents[i][0] for i in r.integers(0, len(pretrain_sents), 64)])
    loss, g = lm_loss_and_grads(p_pre, X, Y, M); clip_grads(g); opt.step(p_pre, g)
    if step % 300 == 0:
        print(f"pretraining step {step}: LM loss {loss:.3f} (perplexity {math.exp(loss):.2f})  ({time.time() - t0:.0f}s)")

### 3.2 Zero-shot: use the language model's probabilities directly
No finetuning at all. GPT-2 did zero-shot Winograd schemas by **comparing probabilities of sequences**: is $P(\dots\text{because the cat was too big}) \geq P(\dots\text{because the hat was too big})$? We do the same: a sentence is judged grammatical if the LM assigns it a higher probability than its verb-flipped twin. (This is also how **minimal-pair benchmarks** like BLiMP evaluate LMs.)

In [ ]:
def sentence_logprob(p, sents):
    X, Y, M = lm_batch(sents)
    xf, _ = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=True, key_mask=M > 0)
    P = softmax(xf @ p["Wout"], -1)
    Bi, Ti = np.meshgrid(np.arange(len(sents)), np.arange(X.shape[1]), indexing="ij")
    return (np.log(P[Bi, Ti, Y] + 1e-12) * M).sum(1)

def zero_shot_accuracy(p, examples):
    sents = [s for s, *_ in examples]
    # the twin of each sentence: the grammatical original if s is the flipped version, else the flipped version
    twins = [orig if s != orig else flip_verb(orig, vp, pl, vpair) for s, _, (orig, vp, pl, vpair) in examples]
    lp_s, lp_t = sentence_logprob(p, sents), sentence_logprob(p, twins)
    pred = (lp_s > lp_t).astype(int)
    return (pred == np.array([y for _, y, _ in examples])).mean()

print(f"zero-shot (LM scoring of minimal pairs): accuracy {zero_shot_accuracy(p_pre, task_test):.1%}")

### 3.3 Step 2: finetune on 200 labels, vs. training from scratch
Following the slides for decoders: *"We can finetune them by training a softmax classifier on the last word's hidden state."* $h_1,\dots,h_T = \text{Decoder}(w_1,\dots,w_T)$, $y\sim Ah_T + b$, where $A$ and $b$ are **randomly initialized** and specified by the downstream task. *Gradients backpropagate through the whole network.* (Note the linear layer hasn't been pretrained and must be learned from scratch.)

We compare three ways to use the same 200 labeled examples:
1. **From scratch:** random backbone, train everything.
2. **Linear probe:** pretrained backbone **frozen**; train only $A, b$.
3. **Full finetuning:** pretrained backbone; train everything (smaller learning rate, so we stay near $\hat\theta$).

In [ ]:
def finetune(p_init, train_keys, lr, steps=300, seed=0):
    p = {k: v.copy() for k, v in p_init.items()}
    r = np.random.default_rng(seed)
    p["A"], p["c"] = r.normal(0, 0.02, (p["tok"].shape[1], 2)), np.zeros(2)
    keys = [k for k in p if k not in ("Wout",)] if train_keys == "all" else ["A", "c"]
    opt = Adam(p, lr=lr, keys=keys)
    for step in range(steps):
        X, M, last, y = cls_batch([task_train[i] for i in r.integers(0, len(task_train), 32)])
        _, g = cls_loss_and_grads(p, X, M, last, y)
        g = {k: g[k] for k in keys}; clip_grads(g); opt.step(p, g)
    X, M, last, y = cls_batch(task_train); _, P_tr = cls_loss_and_grads(p, X, M, last, y, need_grad=False)
    X, M, last, y = cls_batch(task_test); _, P_te = cls_loss_and_grads(p, X, M, last, y, need_grad=False)
    return (P_tr.argmax(1) == cls_batch(task_train)[3]).mean(), (P_te.argmax(1) == y).mean()

p_scratch = init_transformer(V, T_MAX, d=48, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=96, seed=1)
p_scratch["Wout"] = p_pre["Wout"]                     # unused by the classifier
results = {}
t0 = time.time()
results["from scratch (all params)"] = finetune(p_scratch, "all", lr=1e-3)
results["pretrained, linear probe (frozen)"] = finetune(p_pre, "head", lr=1e-2)
results["pretrained, full finetuning"] = finetune(p_pre, "all", lr=3e-4)
print(f"{'method':36s} {'train acc':>9s} {'test acc':>9s}     (200 training labels; {time.time() - t0:.0f}s)")
for name, (tr, te) in results.items():
    print(f"{name:36s} {tr:9.1%} {te:9.1%}")
print(f"{'zero-shot LM scoring (no labels)':36s} {'-':>9s} {zero_shot_accuracy(p_pre, task_test):9.1%}")

**"Pretrain once, finetune many times."** Three lessons from these numbers:
* **From scratch, 200 labels are far too few** to discover subject–verb agreement. The model memorizes the training set (100% train accuracy) and is near chance on the test set.
* **Full finetuning of the pretrained model generalizes far better** from the same 200 labels. The pretrained network already "knows" agreement from unlabeled text, and finetuning only has to learn to *read it out*.
* **The frozen linear probe fails.** The last token's hidden state (at ".") was trained to predict `</s>`, which doesn't depend on whether the verb agreed, so the violation isn't linearly readable there. Full finetuning lets the network re-route that information to the last position. Where you read a representation from matters.

And a fourth, which previews Lecture 10: **zero-shot scoring beats finetuning here.** Scoring uses exactly the knowledge pretraining built ($P(	ext{sentence})$), with nothing new to learn. Finetuning has to learn a new head and new routing from 200 examples. When you can phrase your task as something the pretrained model already does, that's often the best use of it. That's the idea behind prompting.

### 3.4 How do we format inputs to a decoder for finetuning tasks? (GPT, Radford et al. 2018)
Natural language inference: label pairs of sentences as *entailing / contradictory / neutral*. *Premise: The man is in the doorway. Hypothesis: The person is near the door.* GPT formats it as one token sequence for the decoder:

`[START] The man is in the doorway [DELIM] The person is near the door [EXTRACT]`

and the linear classifier is applied to the representation of the `[EXTRACT]` token. This "turn every task into a sequence" trick is the ancestor of prompting (L10).

---
## 4. Model pretraining three ways

The neural architecture influences the type of pretraining, and natural use cases:

| | Architecture | Pretraining | Natural use |
|---|---|---|---|
| **Decoders** | causal self-attention | language modeling (what we've seen so far) | generation; can't condition on future words; **all the biggest models are decoders** |
| **Encoders** | bidirectional self-attention | masked language modeling | gets bidirectional context; good for analysis of text |
| **Encoder–decoders** | both | span corruption / denoising | seq2seq; good parts of both? |

### 4.1 Pretraining encoders: what objective to use?
Encoders get **bidirectional context**, so we can't do language modeling! **Idea:** replace some fraction of words in the input with a special `[MASK]` token; predict these words:
$$h_1,\dots,h_T = \text{Encoder}(w_1,\dots,w_T),\qquad y_i\sim Ah_i + b$$
Only add loss terms from words that are "masked out." If $\tilde x$ is the masked version of $x$, we're learning $p_\theta(x\mid\tilde x)$. This is called **masked LM** (*I [M] to the [M]* → *went*, *store*).

### 4.2 BERT: Bidirectional Encoder Representations from Transformers (Devlin et al. 2018)
Some details about masked LM for BERT:
* Predict a random **15%** of (sub)word tokens.
* Replace the input word with `[MASK]` **80%** of the time.
* Replace the input word with a **random token 10%** of the time.
* Leave the input word **unchanged 10%** of the time (but still predict it!).
* **Why?** It doesn't let the model get complacent and not build strong representations of non-masked words. (No masks are seen at finetuning time!)

In [ ]:
def bert_mask(ids, rng, special_ids, p_select=0.15):
    # Returns (corrupted input ids, prediction targets mask). Targets are the ORIGINAL ids.
    ids = np.array(ids); corrupted = ids.copy()
    candidates = np.array([i not in special_ids for i in ids])
    selected = (rng.random(len(ids)) < p_select) & candidates
    roll = rng.random(len(ids))
    corrupted[selected & (roll < 0.8)] = w2i["[MASK]"]                                  # 80%: [MASK]
    rand_pos = selected & (roll >= 0.8) & (roll < 0.9)                                  # 10%: random token
    corrupted[rand_pos] = rng.integers(4, V, rand_pos.sum())
    return corrupted, selected                                                          # 10%: unchanged

r = np.random.default_rng(0); special = {w2i[t] for t in ["<pad>", "<s>", "</s>", "[MASK]"]}
stats = collections.Counter()
for s, *_ in pretrain_sents[:3000]:
    ids = [w2i[w] for w in s]
    corr, sel = bert_mask(ids, r, special)
    for o, c_, sl in zip(ids, corr, sel):
        stats["tokens"] += 1
        if sl:
            stats["selected"] += 1
            stats["[MASK]" if c_ == w2i["[MASK]"] else ("unchanged" if c_ == o else "random")] += 1
print(f"selected for prediction: {stats['selected'] / stats['tokens']:.1%} of tokens")
for k in ["[MASK]", "random", "unchanged"]:
    print(f"  of those, {k:9s}: {stats[k] / stats['selected']:.1%}")
s0 = pretrain_sents[0][0]; corr, sel = bert_mask([w2i[w] for w in s0], np.random.default_rng(3), special, p_select=0.3)
print("\noriginal :", " ".join(s0)); print("corrupted:", " ".join(vocab[i] for i in corr), "  (30% selection here, for visibility)")

**Training a small "BERT".** Same backbone, but **bidirectional** (no causal mask), trained to reconstruct the selected tokens.

**Why bidirectional context matters, measured.** In our language, the subject's number is often revealed only by the **verb, far to the right**: *the [MASK] that the cat saw near the farmer **sleep***. We mask the subject noun of test sentences and ask whether the model predicts the right **number** (singular vs. plural form). A left-to-right LM at that position has seen only *the* (or *the old*), so it can only guess; the encoder can read the verb.

In [ ]:
def mlm_batch(sents, rng):
    L = max(len(s) for s in sents) + 1
    X = np.zeros((len(sents), L), int); Y = np.zeros((len(sents), L), int); M = np.zeros((len(sents), L)); K = np.zeros((len(sents), L), bool)
    for b, s in enumerate(sents):
        ids = [w2i["<s>"]] + [w2i[w] for w in s]
        corr, sel = bert_mask(ids, rng, special)
        X[b, :len(ids)] = corr; Y[b, :len(ids)] = ids; M[b, :len(ids)] = sel; K[b, :len(ids)] = True
    return X, Y, M, K

def mlm_loss_and_grads(p, X, Y, M, K):
    xf, cache = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=False, key_mask=K)   # bidirectional!
    loss, G, _ = token_ce(xf @ p["Wout"], Y, M)                                          # loss only on selected tokens
    g = backbone_backward(p, G @ p["Wout"].T, cache, N_LAYERS, N_HEADS)
    g["Wout"] = xf.reshape(-1, xf.shape[-1]).T @ G.reshape(-1, G.shape[-1])
    return loss, g

p_bert = init_transformer(V, T_MAX, d=48, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=96, seed=2)
p_bert["Wout"] = np.random.default_rng(2).normal(0, 1 / np.sqrt(48), (48, V))
opt = Adam(p_bert, lr=1e-3); r = np.random.default_rng(0); t0 = time.time()
for step in range(1, 1201):
    batch = [pretrain_sents[i][0] for i in r.integers(0, len(pretrain_sents), 64)]
    loss, g = mlm_loss_and_grads(p_bert, *mlm_batch(batch, r)); clip_grads(g); opt.step(p_bert, g)
    if step % 300 == 0:
        print(f"MLM step {step}: masked-token loss {loss:.3f}  ({time.time() - t0:.0f}s)")

In [ ]:
sg_ids = [w2i[n[0]] for n in NOUNS]; pl_ids = [w2i[n[1]] for n in NOUNS]
probe = [gen_sentence(rng=random.Random(100 + i)) for i in range(500)]

def subject_number_accuracy_encoder(p):
    correct = 0
    for s, sp, vp, pl, _ in probe:
        ids = [w2i["<s>"]] + [w2i[w] for w in s]; ids[sp + 1] = w2i["[MASK]"]
        X = np.array([ids]); xf, _ = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=False, key_mask=np.ones(X.shape, bool))
        P = softmax(xf[0, sp + 1] @ p["Wout"])
        correct += (P[pl_ids].sum() > P[sg_ids].sum()) == bool(pl)
    return correct / len(probe)

def subject_number_accuracy_decoder(p):
    correct = 0
    for s, sp, vp, pl, _ in probe:
        ids = [w2i["<s>"]] + [w2i[w] for w in s[:sp]]                      # only the LEFT context is available
        X = np.array([ids]); xf, _ = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=True, key_mask=np.ones(X.shape, bool))
        P = softmax(xf[0, -1] @ p["Wout"])
        correct += (P[pl_ids].sum() > P[sg_ids].sum()) == bool(pl)
    return correct / len(probe)

print(f"predict the number of a masked SUBJECT noun (needs the verb on the right):")
print(f"  bidirectional encoder (masked LM):   {subject_number_accuracy_encoder(p_bert):.1%}")
print(f"  left-to-right decoder (LM):          {subject_number_accuracy_decoder(p_pre):.1%}   (it can't see the verb)")
avg_dist = np.mean([vp - sp for _, sp, vp, _, _ in probe])
print(f"  (average distance from subject to verb in these sentences: {avg_dist:.1f} words)")

### 4.3 BERT, in numbers
* **Next sentence prediction (NSP):** the pretraining input to BERT was two separate contiguous chunks of text, and BERT was also trained to predict whether one chunk follows the other or is randomly sampled. Later work argued NSP is **not necessary** (Liu et al. 2019).
* Two models were released:
    * **BERT-base:** 12 layers, 768-dim hidden states, 12 attention heads, **110 million** params.
    * **BERT-large:** 24 layers, 1024-dim hidden states, 16 attention heads, **340 million** params.
* Trained on BooksCorpus (800 million words) and English Wikipedia (2,500 million words).
* Pretraining is expensive and impractical on a single GPU: BERT was pretrained with 64 TPU chips for a total of 4 days. **Finetuning is practical and common on a single GPU. "Pretrain once, finetune many times."**
* BERT was massively popular and versatile; finetuning it led to new state-of-the-art results on a broad range of tasks: **QQP** (Quora question pairs, detecting paraphrase questions), **QNLI** (NLI over QA data), **SST-2** (sentiment), **CoLA** (grammatical acceptability; the task we just imitated!), **STS-B** (semantic textual similarity), **MRPC** (Microsoft paraphrase corpus), **RTE** (a small NLI corpus).

**Limitations of pretrained encoders.** If your task involves **generating sequences**, consider using a pretrained decoder: BERT and other pretrained encoders don't naturally lead to nice autoregressive (1-word-at-a-time) generation methods. (*Iroh goes to [MASK] tasty tea* → make/brew/craft: fine for one blank, awkward for writing a paragraph.)

**Extensions of BERT** (RoBERTa, SpanBERT, +++). Generally accepted improvements to the BERT recipe:
* **RoBERTa:** mainly just **train BERT for longer and remove next-sentence prediction!** A takeaway: **more compute and more data can improve pretraining even when not changing the underlying Transformer encoder.**
* **SpanBERT:** masking **contiguous spans** of words makes a harder, more useful pretraining task (*It's [MASK] [MASK] [MASK] good* → *irr## esi## sti## bly*, rather than a single subword whose neighbors give it away).

---
## 5. Pretraining encoder–decoders

For encoder–decoders, we could do something like language modeling, where a **prefix** of every input is provided to the encoder and is not predicted:
$$h_1,\dots,h_T = \text{Encoder}(w_1,\dots,w_T),\qquad h_{T+1},\dots,h_{2T} = \text{Decoder}(w_{T+1},\dots,w_{2T}, h_1,\dots,h_T),\qquad y_i\sim Ah_i+b,\ i>T$$
The encoder benefits from bidirectional context; the decoder trains the whole model through language modeling.

What Raffel et al. (2018) found to work best was **span corruption**. Their model: **T5**. Replace different-length spans from the input with **unique placeholders**; decode out the spans that were removed! This is implemented in **text preprocessing**: it's still an objective that looks like language modeling at the decoder side.

*Original text:* Thank you for inviting me to your party last week.  
*Inputs:* Thank you `<X>` me to your party `<Y>` week.  
*Targets:* `<X>` for inviting `<Y>` last `<Z>`

In [ ]:
def span_corrupt(tokens, spans):
    # spans: list of (start, length) to remove. Returns (encoder input, decoder target) with T5 sentinels.
    sentinels = [f"<extra_id_{i}>" for i in range(len(spans) + 1)]
    inp, tgt, i = [], [], 0
    for k, (start, length) in enumerate(sorted(spans)):
        inp += tokens[i:start] + [sentinels[k]]
        tgt += [sentinels[k]] + tokens[start:start + length]
        i = start + length
    inp += tokens[i:]
    tgt += [sentinels[len(spans)]]                       # final sentinel marks the end
    return inp, tgt

toks = "Thank you for inviting me to your party last week .".split()
inp, tgt = span_corrupt(toks, [(2, 2), (8, 1)])
print("input :", " ".join(inp)); print("target:", " ".join(tgt))

def random_span_corrupt(tokens, rng, noise=0.15, mean_span=3):
    n_noise = max(1, round(len(tokens) * noise)); spans, covered = [], set()
    while sum(l for _, l in spans) < n_noise:
        length = max(1, int(rng.poisson(mean_span - 1)) + 1); start = int(rng.integers(0, max(1, len(tokens) - length)))
        if covered.isdisjoint(range(start - 1, start + length + 1)):
            spans.append((start, length)); covered.update(range(start, start + length))
    return span_corrupt(tokens, spans)

inp, tgt = random_span_corrupt(pretrain_sents[5][0] + pretrain_sents[6][0], np.random.default_rng(1))
print("\nrandom 15% span corruption of a toy sentence:\n input :", " ".join(inp), "\n target:", " ".join(tgt))

Raffel et al. found encoder–decoders to work better than decoders for their tasks, and span corruption (denoising) to work better than language modeling. **A fascinating property of T5:** it can be finetuned to answer a wide range of questions, **retrieving knowledge from its parameters** ("closed-book" QA on Natural Questions, WebQuestions, TriviaQA), with accuracy increasing steadily from 220M to 770M to 3B to 11B parameters.

---
## 6. Pretraining decoders

When using language-model-pretrained decoders, we can **ignore** that they were trained to model $p(w_t\mid w_{1:t-1})$:
* **As a classifier:** train a softmax classifier on the last word's hidden state (what we did in §3.3); $A, b$ are new.
* **As a generator:** finetune their $p_\theta(w_t\mid w_{1:t-1})$ directly. This is natural for tasks where the output is a sequence with a vocabulary like that at pretraining time: **dialogue** (context = dialogue history), **summarization** (context = document). Here $A, b$ were pretrained in the language model.

**GPT (Radford et al. 2018)** was a big success in pretraining a decoder: a Transformer decoder with **12 layers, 117M parameters**, 768-dim hidden states, 3072-dim feed-forward hidden layers, BPE with 40,000 merges, trained on BooksCorpus (over 7,000 unique books, with long spans of contiguous text for learning long-distance dependencies). The acronym "GPT" never showed up in the original paper; it could stand for "Generative PreTraining" or "Generative Pretrained Transformer."

**GPT-2** (1.5B parameters, more data) produced relatively convincing samples of natural language. The slides show its famous completion of a prompt about stolen nuclear materials in Cincinnati: a fluent, plausible, and **entirely fabricated** news story, complete with invented quotes from an invented Energy Secretary.

### 6.1 GPT-3, in-context learning, and very large models
So far, we've interacted with pretrained models in two ways: **sample** from the distributions they define (maybe providing a prompt), or **finetune** them on a task. **Very large language models seem to perform some kind of learning without gradient steps simply from examples you provide within their contexts.** GPT-3 (175 billion parameters; the largest T5 had 11 billion) is the canonical example. The in-context examples seem to **specify the task**, and the conditional distribution mocks performing it:

```
thanks -> merci
hello -> bonjour
mint -> menthe
otter ->            (model continues: "loutre")
```

L10 builds a tiny model that learns to do this kind of task inference, and covers instruction tuning (ChatGPT/GPT-4).

---
## 7. Why scale? Scaling laws

* **Empirical observation:** scaling up models leads to **reliable gains in perplexity** (Kaplan et al. 2020): test loss falls as a power law in parameters $N$, data $D$, and compute $C$, e.g. $L(N)\approx(N_c/N)^{\alpha_N}$.
* Scaling laws help identify **model size vs. data tradeoffs**, and exist for many other architecture decisions. **Predictable scaling helps us make intelligent decisions** (try ideas small, extrapolate).

**Scaling efficiency: how do we best use our compute?** Roughly, the cost of training a large Transformer scales as **parameters × tokens**: $C\approx 6ND$ FLOPs (2 for the forward pass, 4 for the backward pass, per parameter per token). GPT-3 was 175B parameters trained on 300B tokens. **Did OpenAI strike the right parameter–token tradeoff? No.** The **Chinchilla** paper (Hoffmann et al. 2022) fit the loss as
$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$
and minimized it subject to $6ND = C$. Their **70B-parameter Chinchilla** (1.4T tokens) is better than much larger models trained on fewer tokens (Gopher 280B, GPT-3 175B), using the same compute as Gopher. Let's use their published fit ("approach 3": $E=1.69, A=406.4, B=410.7, \alpha=0.34, \beta=0.28$):

In [ ]:
E_, A_, B_, al, be = 1.69, 406.4, 410.7, 0.34, 0.28
def chinchilla_loss(N, D):
    return E_ + A_ / N ** al + B_ / D ** be

def compute_optimal(C):
    # Minimize L(N, D) s.t. 6ND = C. Closed form: N_opt = G (C/6)^(be/(al+be)), D_opt = (C/6)/N_opt.
    G = (al * A_ / (be * B_)) ** (1 / (al + be))
    N = G * (C / 6) ** (be / (al + be))
    return N, C / 6 / N

models = {"GPT-3": (175e9, 300e9), "Gopher": (280e9, 300e9), "Chinchilla": (70e9, 1.4e12), "LLaMA-2 7B": (7e9, 2e12)}
print(f"{'model':12s} {'params':>8s} {'tokens':>8s} {'tokens/param':>12s} {'FLOPs (6ND)':>12s} {'predicted loss':>14s}")
for name, (N, D) in models.items():
    print(f"{name:12s} {N / 1e9:7.0f}B {D / 1e9:7.0f}B {D / N:12.1f} {6 * N * D:12.2e} {chinchilla_loss(N, D):14.3f}")

C = 6 * 280e9 * 300e9
N_opt, D_opt = compute_optimal(C)
print(f"\nWith Gopher's compute ({C:.2e} FLOPs), the fit says: N = {N_opt / 1e9:.0f}B params, D = {D_opt / 1e12:.2f}T tokens "
      f"({D_opt / N_opt:.0f} tokens/param), loss {chinchilla_loss(N_opt, D_opt):.3f} vs Gopher's {chinchilla_loss(280e9, 300e9):.3f}")

# numerical check of the closed form: scan N along the compute constraint
Ns = np.logspace(9, 12.5, 2000); Ls = chinchilla_loss(Ns, C / 6 / Ns)
print(f"numerical minimum along 6ND = C: N = {Ns[Ls.argmin()] / 1e9:.0f}B")

Cs = np.logspace(19, 26, 50); opt = [compute_optimal(c) for c in Cs]
plt.figure(figsize=(7, 4))
plt.loglog(Cs, [n for n, _ in opt], label="compute-optimal params N")
plt.loglog(Cs, [d for _, d in opt], label="compute-optimal tokens D")
for name, (N, D) in models.items():
    plt.scatter(6 * N * D, N, marker="x"); plt.annotate(name, (6 * N * D, N), fontsize=8)
plt.xlabel("training compute (FLOPs)"); plt.legend(); plt.title("Chinchilla scaling (approach-3 fit)"); plt.show()

With this fit, GPT-3 and Gopher are far on the "too many parameters, too few tokens" side. (Chinchilla's three estimation approaches disagree somewhat; approaches 1 and 2 give the famous rule of thumb of **~20 tokens per parameter**, and approach 3 gives somewhat smaller, more data-hungry models at large budgets. All three agreed that GPT-3-era models were undertrained.)

**Modern observation: train a big model that's not fully converged**, or rather, train a *smaller* model on far *more* tokens than "compute-optimal" (LLaMA-2 7B: ~290 tokens/param). Chinchilla optimizes *training* compute; if a model will serve billions of queries, a smaller model that's cheaper at **inference** time is worth the extra training compute.

---
## 8. What do we think pretraining is teaching?

There's increasing evidence that pretrained models learn a wide variety of things about the statistical properties of language. Taking the examples from §2.1: **trivia** (*Stanford is located in Palo Alto*), **syntax** (*I put **the** fork*), **coreference** (*over **her** shoulder*), **lexical semantics/topic** (*fish, turtles, seals, and **sharks***), **sentiment** (*the movie was **bad***), **some reasoning** (this is harder: *Zuko left the **kitchen***), and **some basic arithmetic** (they don't learn the Fibonacci sequence). In our toy experiment, pretraining alone taught the model long-distance agreement well enough to judge grammaticality zero-shot.

But models also learn things we **don't** want:
* Models learn, and can exacerbate, **racism, sexism, all manner of bad biases**: the prejudices of the humans who write online.
* They sometimes **memorize copyrighted material**.
* **Membership inference** lets you recover parts of the training data, and sometimes that is **semi-private material** from the web (addresses, emails) (Carlini et al. 2021).

### Three types of architectures for pretraining: the verdict
* **Decoders:** language models; **scale well**; best to generate from; **have won** as to what people build.
* **Encoders:** bidirectional context; **good if only doing analysis of text** (better than decoders at that, for their size); e.g., embedding models for retrieval and classifiers are still often BERT-style.
* **Encoder–decoders:** good parts of both? Some evidence they're better for NLU (Tay et al. 2022, UL2).

---
## 9. The real thing with Hugging Face (optional)

These cells download real pretrained models (~250–500MB each). They show the three families: BERT filling a mask with bidirectional context, GPT-2 scoring minimal pairs zero-shot (exactly §3.2, with a real model), and T5's sentinel tokens.

In [ ]:
# [Optional] needs: pip install transformers torch   (and internet)
try:
    from transformers import pipeline
    fill = pipeline("fill-mask", model="bert-base-uncased")
    for text in ["Stanford University is located in [MASK], California.",
                 "I put [MASK] fork down on the table.",
                 "The woman walked across the street, checking for traffic over [MASK] shoulder."]:
        print(text, "->", [(c["token_str"], round(c["score"], 3)) for c in fill(text)[:3]])
except Exception as e:
    print("Skipping BERT demo:", e)

In [ ]:
# [Optional] GPT-2 as a zero-shot grammaticality judge (minimal pairs), and its BPE tokenizer.
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained("gpt2"); gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    def logprob(text):
        ids = tok(text, return_tensors="pt").input_ids
        with torch.no_grad():
            out = gpt2(ids, labels=ids)
        return -out.loss.item() * (ids.shape[1] - 1)          # total log-prob of tokens 2..n
    pairs = [("The keys to the cabinet are on the table.", "The keys to the cabinet is on the table."),
             ("The author that the guards like laughs.", "The author that the guards like laugh."),
             ("The cat couldn't fit into the hat because the cat was too big.", "The cat couldn't fit into the hat because the hat was too big.")]
    for good, bad in pairs:
        print(f"{'✓' if logprob(good) > logprob(bad) else '✗'}  prefers: {good if logprob(good) > logprob(bad) else bad}")
    print("\nGPT-2 byte-level BPE:", [tok.decode([i]) for i in tok("Transformerify taaaaasty laern").input_ids])
except Exception as e:
    print("Skipping GPT-2 demo:", e)

In [ ]:
# [Optional] T5 fills sentinel spans (its pretraining task).
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    t5_tok = AutoTokenizer.from_pretrained("t5-small"); t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
    ids = t5_tok("Thank you <extra_id_0> me to your party <extra_id_1> week.", return_tensors="pt").input_ids
    print(t5_tok.decode(t5.generate(ids, max_new_tokens=12)[0]))
except Exception as e:
    print("Skipping T5 demo:", e)

---
## 10. Summary

| Topic | Takeaway |
|---|---|
| Subwords | BPE: start from characters, repeatedly merge the most frequent adjacent pair; no more UNK; byte-level BPE handles any string |
| Contextual meaning | Firth 1935: meaning is contextual (*I record the record*); pretrain whole models, not just word vectors |
| Pretrain → finetune | Learn general things from lots of unlabeled text, then adapt with few labels; SGD stays near $\hat\theta$ |
| Measured (toy) | With 200 labels: from scratch ≈ chance on test, pretrained + finetuned much better, zero-shot LM scoring best |
| Encoders (BERT) | Masked LM: 15% of tokens; 80% [MASK] / 10% random / 10% unchanged; bidirectional; not natural generators |
| Encoder–decoders (T5) | Span corruption with sentinel tokens; closed-book QA improves with scale |
| Decoders (GPT) | LM pretraining; finetune as classifier (last token) or as generator; GPT-3 in-context learning |
| Scaling | $C\approx6ND$; loss is predictable; Chinchilla: GPT-3/Gopher were undertrained; today: overtrain small models for cheap inference |
| Risks | Bias, memorization (copyright, privacy), membership inference |

### Exercises
1. Implement **WordPiece**-style merging: choose the pair maximizing $\frac{\text{count}(ab)}{\text{count}(a)\,\text{count}(b)}$ instead of $\text{count}(ab)$. How do the merges differ on the corpus above?
2. Vary the number of labeled examples (25, 50, 100, 400, 1600) in §3.3 and plot test accuracy for "from scratch" vs. "full finetuning". Where do the curves meet?
3. Finetune the **MLM encoder** (`p_bert`) on the grammaticality task (classify from the `<s>` token's hidden state, bidirectionally). Is it better or worse than the decoder?
4. Change BERT's masking to 100% `[MASK]` (no random/unchanged). Then evaluate the encoder on *unmasked* input. Why might the 80/10/10 rule help here?
5. Use the Chinchilla fit to find the compute-optimal model for $10^{22}$ FLOPs, and compute how much worse a model 10× larger (with 10× fewer tokens) would be.

### References
* Sennrich, Haddow & Birch 2016, *Neural Machine Translation of Rare Words with Subword Units* (BPE).
* Dai & Le 2015, *Semi-supervised Sequence Learning*; Radford et al. 2018 (GPT), 2019 (GPT-2); Brown et al. 2020 (GPT-3).
* Devlin et al. 2018 (BERT); Liu et al. 2019 (RoBERTa); Joshi et al. 2020 (SpanBERT).
* Raffel et al. 2020, *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5).
* Kaplan et al. 2020 (scaling laws); Hoffmann et al. 2022 (Chinchilla).
* Carlini et al. 2021, *Extracting Training Data from Large Language Models*.
* Warstadt et al. 2020, *BLiMP: The Benchmark of Linguistic Minimal Pairs*.

**Next:** Lecture 10, from language models to assistants: prompting, instruction finetuning, RLHF, and DPO.